In [9]:
import pandas as pd
import json
import spacy

In [10]:
nlp = spacy.load("en_core_sci_sm")
print("Model loaded successfully!")

Model loaded successfully!


In [11]:
df = pd.read_csv("../data/transcriptions/train.csv")

print(df.head())
print("Total samples:", len(df))

                            file  \
0  1249120_44176037_58635902.wav   
1  1249120_44176037_96532055.wav   
2  1249120_44220382_41016711.wav   
3  1249120_44188922_16615356.wav   
4  1249120_44246595_60938320.wav   

                                                text  
0   I was diagnosed with pneumonia. I can't breat...  
1   I feel muscle pain every time I make an extra...  
2      I have a back pain since I fell on the floor.  
3   When I try to be warm and with more clothes I...  
4                             My ear hurts me badly.  
Total samples: 381


In [12]:
def label_text(text):
    doc = nlp(text)

    tokens = [token.text.lower() for token in doc]
    labels = ["O"] * len(tokens)

    # --- 1. scispaCy entities ---
    for ent in doc.ents:
        start = ent.start
        end = ent.end

        labels[start] = "B-SYMPTOM"
        for i in range(start + 1, end):
            labels[i] = "I-SYMPTOM"

    # --- 2. Additional rules ---
    for i, token in enumerate(tokens):

        # X pain (stomach pain, chest pain, etc.)
        if token == "pain" and i > 0:
            labels[i] = "I-SYMPTOM"
            labels[i-1] = "B-SYMPTOM"

        # feeling + next word
        if token == "feeling" and i < len(tokens) - 1:
            labels[i+1] = "B-SYMPTOM"

        # mental / general symptoms
        if token in ["tensed", "stressed", "anxious", "weak", "tired", "dizzy"]:
            labels[i] = "B-SYMPTOM"

        # functional issues
        if token in ["cannot", "unable"]:
            labels[i] = "B-SYMPTOM"

        # duration
        if token in ["yesterday", "today", "days", "weeks", "night"]:
            labels[i] = "B-DURATION"

    return tokens, labels

In [13]:
data = []

for text in df["text"]:
    tokens, labels = label_text(text)

    data.append({
        "tokens": tokens,
        "ner_tags": labels
    })

print("Sample output:")
print(data[0])

Sample output:
{'tokens': [' ', 'i', 'was', 'diagnosed', 'with', 'pneumonia', '.', 'i', 'ca', "n't", 'breathe', 'easily', '.'], 'ner_tags': ['O', 'O', 'O', 'B-SYMPTOM', 'O', 'B-SYMPTOM', 'O', 'O', 'O', 'O', 'O', 'O', 'O']}


In [15]:
with open("../data/annotated/train.json", "w") as f:
    json.dump(data, f, indent=2)

print("✅ train.json created successfully!")

✅ train.json created successfully!
